Baseline Artistes

In [1]:
%pip install python-dotenv
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

import os
from dotenv import load_dotenv
from pathlib import Path
import psycopg2

In [3]:

# Load .env from parent directory
load_dotenv(dotenv_path=Path("../.env"))

# Connect
conn = psycopg2.connect(
    dbname=os.getenv("DATABASE"),
    user=os.getenv("DB_USERNAME"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("POSTGRES"),
    port=5432
)

print(f"Connection successful: {conn}")

query = """
SELECT
    a.id AS artist_id,
    a.name AS artist_name,
    atype.name AS artist_type,
    ar.name AS area_name,

    STRING_AGG(DISTINCT t.name, ' ') AS tags,
    STRING_AGG(DISTINCT g.name, ' ') AS genres,

    SUM(GREATEST(art.count, 0)) AS tag_count_sum

FROM musicbrainz.artist a

LEFT JOIN musicbrainz.artist_type atype
    ON a.type = atype.id

LEFT JOIN musicbrainz.area ar
    ON a.area = ar.id

LEFT JOIN musicbrainz.artist_tag art
    ON a.id = art.artist

LEFT JOIN musicbrainz.tag t
    ON art.tag = t.id

LEFT JOIN musicbrainz.genre g
    ON LOWER(t.name) = LOWER(g.name)

WHERE a.name IS NOT NULL
  AND art.count IS NOT NULL
  AND LOWER(a.name) != 'various artists'

GROUP BY
    a.id,
    a.name,
    atype.name,
    ar.name

HAVING SUM(GREATEST(art.count, 0)) > 0

ORDER BY tag_count_sum DESC

"""

df = pd.read_sql(query, conn)
conn.close()

Connection successful: <connection object at 0x790474365bc0; dsn: 'user=musicbrainz password=xxx dbname=musicbrainz_db host=34.14.1.32 port=5432', closed: 0>


/tmp/ipykernel_75776/587006444.py:60: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## Description du dataframe utilisé pour la baseline artiste

Ce dataframe est construit à partir de la base MusicBrainz afin de créer une première baseline de recommandation d’artistes.
L’objectif est de disposer d’une ligne par artiste, enrichie avec des informations textuelles, catégorielles et numériques pouvant être utilisées dans un modèle de recommandation basé sur **TF-IDF** et **KNN / Nearest Neighbors**.

Chaque ligne du dataframe représente donc un artiste suffisamment renseigné dans MusicBrainz, notamment avec des tags exploitables.

### Colonnes issues de la requête SQL

| Colonne         |      Type attendu | Description                                                                                                                                                                                                    | Utilisation dans le modèle                                                                                     |
| --------------- | ----------------: | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------- |
| `artist_id`     |            entier | Identifiant interne de l’artiste dans la base MusicBrainz. Il sert de clé unique pour retrouver un artiste et éviter les doublons.                                                                             | Utilisé comme identifiant, mais pas comme feature d’entraînement.                                              |
| `artist_name`   |             texte | Nom principal de l’artiste, du groupe ou de l’entité musicale. Exemple : `Eminem`, `Radiohead`, `Daft Punk`.                                                                                                   | Sert surtout à rechercher l’artiste de départ. Peut être utilisé avec prudence dans les features textuelles.   |
| `artist_type`   |         catégorie | Type d’artiste selon MusicBrainz : personne, groupe, orchestre, chœur, etc.                                                                                                                                    | Variable catégorielle encodée avec `OneHotEncoder`.                                                            |
| `area_name`     | catégorie / texte | Zone géographique associée à l’artiste lorsqu’elle est renseignée : pays, ville ou région.                                                                                                                     | Variable catégorielle pouvant aider à rapprocher des artistes issus d’un même espace culturel ou géographique. |
| `tags`          |             texte | Ensemble des tags MusicBrainz associés à l’artiste. Les tags peuvent contenir des styles musicaux, des indications culturelles ou des descripteurs variés. Exemple : `rap`, `hip hop`, `french`, `electronic`. | Colonne textuelle principale vectorisée avec `TF-IDF`.                                                         |
| `genres`        |             texte | Sous-ensemble des tags correspondant à des genres reconnus dans la table `genre` de MusicBrainz. Exemple : `hip hop`, `rock`, `jazz`, `electronic`.                                                            | Colonne textuelle importante, également vectorisée avec `TF-IDF`.                                              |
| `tag_count_sum` |         numérique | Somme des poids des tags associés à l’artiste. Elle donne une indication du niveau de renseignement de l’artiste dans MusicBrainz.                                                                             | Feature numérique optionnelle. Elle peut être nettoyée puis transformée avant entraînement.                    |

### Colonnes créées pendant le preprocessing

| Colonne           | Type attendu | Description                                                                                                                                                                      | Utilisation dans le modèle                            |
| ----------------- | -----------: | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------- |
| `tag_count_clean` |    numérique | Version nettoyée de `tag_count_sum`, dans laquelle les valeurs négatives sont ramenées à 0. Cela évite que des valeurs atypiques perturbent le modèle.                           | Sert de base à la transformation logarithmique.       |
| `tag_count_log`   |    numérique | Transformation logarithmique de `tag_count_clean` avec `log1p`. Cette transformation réduit l’effet des très grandes valeurs.                                                    | Feature numérique standardisée avec `StandardScaler`. |
| `text_features`   |        texte | Colonne textuelle construite en concaténant principalement `genres` et `tags`. Les genres peuvent être volontairement répétés pour leur donner plus de poids dans la similarité. | Colonne vectorisée avec `TfidfVectorizer`.            |

### Gestion des valeurs manquantes

Certaines colonnes peuvent contenir des valeurs manquantes, notamment `area_name` ou `genres`.

Le choix retenu pour la baseline est de ne pas supprimer brutalement ces lignes, mais de remplacer les valeurs manquantes :

* `artist_type` → `"Unknown"`
* `area_name` → `"Unknown"`
* `tags` → chaîne vide `""`
* `genres` → chaîne vide `""`
* `artist_name` → chaîne vide `""`, même si les artistes sans nom sont normalement filtrés en amont

Cette stratégie permet de conserver un maximum d’artistes tout en évitant les erreurs lors de l’encodage.

### Gestion des doublons

Avant l’entraînement, plusieurs contrôles sont effectués :

* recherche des doublons complets ;
* recherche des doublons sur `artist_id` ;
* recherche éventuelle des doublons sur `artist_name`.

Le dataframe final doit idéalement contenir une seule ligne par `artist_id`.

### Features utilisées pour la baseline

Pour cette première baseline, les features utilisées sont séparées en trois groupes :

#### Features textuelles

* `text_features`

Cette colonne est vectorisée avec `TfidfVectorizer`.
Elle permet de transformer les tags et genres en vecteurs numériques exploitables par le modèle.

#### Features catégorielles

* `artist_type`
* `area_name`

Ces colonnes sont encodées avec `OneHotEncoder`, afin de transformer les catégories textuelles en variables numériques.

#### Feature numérique

* `tag_count_log`

Cette colonne est standardisée avec `StandardScaler`.
Elle représente le niveau de richesse des tags associés à un artiste, mais avec une transformation qui limite l’impact des valeurs extrêmes.

### Objectif du modèle

Le modèle utilisé est une baseline de recommandation non supervisée.

Il ne cherche pas à prédire une cible comme dans une classification ou une régression.
Il cherche plutôt à identifier les artistes les plus proches d’un artiste donné à partir de leurs métadonnées.

La logique est la suivante :

1. transformer les informations textuelles avec TF-IDF ;
2. encoder les variables catégorielles ;
3. standardiser la variable numérique ;
4. construire une représentation vectorielle de chaque artiste ;
5. utiliser un modèle KNN avec une distance cosinus ;
6. retourner les artistes les plus proches de l’artiste de départ.

Cette approche permet de produire une première recommandation simple, interprétable et adaptée aux données MusicBrainz.


In [4]:
df = pd.read_csv("../raw_data/artist_reco_dataset_sample.csv")

In [5]:
df.head(10)

,artist_id,artist_name,artist_type,area_name,tags,genres,tag_count_sum
0,1,Various Artists,Other,NaN,1 16 217528 150 13231 25894 39556 62358 75112 ...,ambient berlin school binaural beats bluegrass...,-1443
1,4,Massive Attack,Group,United Kingdom,alternative dance alternative rock ambient bri...,alternative dance alternative rock ambient chi...,78
2,6,Apartment 26,Group,United Kingdom,alternative metal british electronic english j...,alternative metal electronic jazz metal nu metal,10
3,7,Dr. Evil,Character,NaN,bogus artist fictional fictitious artist ficti...,NaN,4
4,9,Robert Miles,Person,Italy,ambient ambient house downtempo dream trance e...,ambient ambient house downtempo dream trance e...,21
5,10,Vincent Gallo,Person,United States,american warp,NaN,2
6,11,Squirrel Nut Zippers,Group,United States,american jazz swing swing revival,jazz swing swing revival,10
7,12,Giant Sand,Group,United States,alternative country alternative rock american ...,alternative country alternative rock country d...,5
8,15,Éric Serra,Person,France,european film composer french soundtrack,NaN,4
9,16,William S. Burroughs,Person,United States,american beat poetry,beat poetry,1


In [6]:
df.shape

(256076, 7)

In [7]:
#Nan et doublons

print("\nValeurs manquantes :")
print(df.isna().sum())

print("\nDoublons complets :")
print(df.duplicated().sum())

print("\nDoublons artist_id :")
print(df.duplicated(subset=["artist_id"]).sum())

print("\nDoublons artist_name :")
print(df.duplicated(subset=["artist_name"]).sum())

print("\nStatistiques tag_count_sum :")
print(df["tag_count_sum"].describe())


Valeurs manquantes :
artist_id            0
artist_name          7
artist_type      14589
area_name        40496
tags                 0
genres           53663
tag_count_sum        0
dtype: int64

Doublons complets :
0

Doublons artist_id :
0

Doublons artist_name :
12128

Statistiques tag_count_sum :
count    256076.000000
mean          3.220563
std          40.499278
min       -1443.000000
25%           1.000000
50%           2.000000
75%           4.000000
max       20296.000000
Name: tag_count_sum, dtype: float64


In [8]:
df_clean = df.copy()

df_clean["artist_name"] = df_clean["artist_name"].fillna("")
df_clean["artist_type"] = df_clean["artist_type"].fillna("Unknown")
df_clean["area_name"] = df_clean["area_name"].fillna("Unknown")
df_clean["tags"] = df_clean["tags"].fillna("")
df_clean["genres"] = df_clean["genres"].fillna("")

In [9]:
#retirer les doublons

df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset=["artist_id"])

In [10]:
# pour tag_count_sum, pas de valeur brute . Variable nettoyée

df_clean["tag_count_clean"] = df_clean["tag_count_sum"].clip(lower=0)
df_clean["tag_count_log"] = np.log1p(df_clean["tag_count_clean"])

In [11]:
#exclusion des "various artitsts"
df_clean = df_clean[
    df_clean["artist_name"].str.lower() != "various artists"
].copy()

In [12]:
df_clean["text_features"] = (
    df_clean["genres"] + " " +
    df_clean["genres"] + " " +  # on répète volontairement les genres pour leur donner plus de poids
    df_clean["tags"]
)

#attention bruit des tags

In [13]:
# Préprocessing TF-IDF + cat + num

text_feature = "text_features"

categorical_features = [
    "artist_type",
    "area_name"
]

numeric_features = [
    "tag_count_log"
]

In [14]:
text_transformer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_transformer, text_feature),
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer, numeric_features)
    ],
    remainder="drop"
)

In [15]:
X = df_clean[
    [text_feature] + categorical_features + numeric_features
]

X_processed = preprocessor.fit_transform(X)

X_processed.shape

(256072, 10423)

In [16]:
knn_model = NearestNeighbors(
    n_neighbors=10,
    metric="cosine",
    algorithm="brute"
)

knn_model.fit(X_processed)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [17]:
# fonction de recommandation d'artiste

def recommend_artists(
    artist_name=None,
    artist_id=None,
    top_n=5,
    genre_filter=None,
    blacklist_artist_ids=None
):
    """
    Recommande des artistes proches à partir d'un artiste de départ.

    Paramètres :
    - artist_name : nom de l'artiste de départ
    - artist_id : id MusicBrainz interne de l'artiste
    - top_n : nombre de recommandations
    - genre_filter : filtre optionnel sur un genre, ex: "rap"
    - blacklist_artist_ids : liste d'artistes à exclure
    """

    if blacklist_artist_ids is None:
        blacklist_artist_ids = []

    if artist_id is not None:
        matches = df_clean[df_clean["artist_id"] == artist_id]
    elif artist_name is not None:
        matches = df_clean[
            df_clean["artist_name"].str.lower() == artist_name.lower()
        ]
    else:
        raise ValueError("Tu dois fournir artist_name ou artist_id.")

    if matches.empty:
        return []

    seed_index = matches.index[0]
    seed_position = df_clean.index.get_loc(seed_index)

    query_vector = X_processed[seed_position]

    n_neighbors = min(len(df_clean), top_n + 30)

    distances, indices = knn_model.kneighbors(
        query_vector,
        n_neighbors=n_neighbors
    )

    recommendations = []

    for distance, idx in zip(distances[0], indices[0]):
        row = df_clean.iloc[idx]

        # Exclure l'artiste lui-même
        if row["artist_id"] == df_clean.iloc[seed_position]["artist_id"]:
            continue

        # Exclure les artistes blacklistés
        if row["artist_id"] in blacklist_artist_ids:
            continue

        # Filtre optionnel sur genre ou tags
        if genre_filter is not None:
            genre_filter_lower = genre_filter.lower()

            genres = str(row["genres"]).lower()
            tags = str(row["tags"]).lower()

            if genre_filter_lower not in genres and genre_filter_lower not in tags:
                continue

        recommendations.append({
            "artist_id": int(row["artist_id"]),
            "artist_name": row["artist_name"],
            "artist_type": row["artist_type"],
            "area_name": row["area_name"],
            "genres": row["genres"],
            "similarity_score": round(1 - distance, 4)
        })

        if len(recommendations) >= top_n:
            break

    return recommendations

In [18]:
#Test

# df_clean[["artist_id", "artist_name", "genres", "tags"]].head(20)

result_0 = recommend_artists(
    artist_name="Eminem",
    top_n=5
)
result_0

[{'artist_id': 762201,
  'artist_name': 'Kendrick Lamar',
  'artist_type': 'Person',
  'area_name': 'United States',
  'genres': 'alternative hip hop boom bap conscious hip hop dance disco experimental hip hop funk future bass gangsta rap hardcore hip hop hip hop house jazz jazz rap neo soul political hip hop pop pop rap r&b trap west coast hip hop',
  'similarity_score': np.float64(0.9912)},
 {'artist_id': 127482,
  'artist_name': 'Ye',
  'artist_type': 'Person',
  'area_name': 'United States',
  'genres': 'art pop chipmunk soul christian hip hop conscious hip hop contemporary r&b deep house electropop experimental hip hop future bass gospel hardcore hip hop hip hop house pop pop rap r&b southern hip hop trap',
  'similarity_score': np.float64(0.9892)},
 {'artist_id': 20544,
  'artist_name': '50 Cent',
  'artist_type': 'Person',
  'area_name': 'United States',
  'genres': 'east coast hip hop gangsta rap hardcore hip hop hip hop pop rap southern hip hop trap',
  'similarity_score': np.

Pipeline V2

In [19]:
df_clean = df.copy()

df_clean["artist_name"] = df_clean["artist_name"].fillna("")
df_clean["artist_type"] = df_clean["artist_type"].fillna("Unknown")
df_clean["area_name"] = df_clean["area_name"].fillna("Unknown")
df_clean["tags"] = df_clean["tags"].fillna("")
df_clean["genres"] = df_clean["genres"].fillna("")

df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset=["artist_id"])

df_clean = df_clean[
    df_clean["artist_name"].str.lower() != "various artists"
].copy()

df_clean["tag_count_clean"] = df_clean["tag_count_sum"].clip(lower=0)

df_clean = df_clean[df_clean["tag_count_clean"] > 0].copy()

In [20]:
# on retire du modèle 0 : artist_id, artist_name, area_name, tag_count_sum


genre_feature = "genres"
tag_feature = "tags"

categorical_features = [
    "artist_type"
]

X = df_clean[
    [genre_feature, tag_feature] + categorical_features
]

In [21]:
#cette fois, pondérer l'importance des features par rapport à d'autres
# genres : 3.0 → information musicale la plus propre
# tags : 1.0 → information riche mais plus bruitée
# artist_type : 0.3 → utile mais secondaire

genre_transformer = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2)
)

tag_transformer = TfidfVectorizer(
    max_features=7000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("genres", genre_transformer, genre_feature),
        ("tags", tag_transformer, tag_feature),
        ("artist_type", categorical_transformer, categorical_features)
    ],
    transformer_weights={
        "genres": 3.0,
        "tags": 1.0,
        "artist_type": 0.3
    },
    remainder="drop"
)

In [22]:
X_processed = preprocessor.fit_transform(X)

knn_model = NearestNeighbors(
    n_neighbors=20,
    metric="cosine",
    algorithm="brute"
)

knn_model.fit(X_processed)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",20
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [23]:
def recommend_artists(
    artist_name=None,
    artist_id=None,
    top_n=5,
    genre_filter=None,
    min_tag_count=0,
    blacklist_artist_ids=None
):
    if blacklist_artist_ids is None:
        blacklist_artist_ids = []

    if artist_id is not None:
        matches = df_clean[df_clean["artist_id"] == artist_id]
    elif artist_name is not None:
        matches = df_clean[
            df_clean["artist_name"].str.lower() == artist_name.lower()
        ]
    else:
        raise ValueError("Il faut fournir artist_name ou artist_id.")

    if matches.empty:
        return []

    seed_index = matches.index[0]
    seed_position = df_clean.index.get_loc(seed_index)

    query_vector = X_processed[seed_position]

    n_neighbors = min(len(df_clean), top_n + 50)

    distances, indices = knn_model.kneighbors(
        query_vector,
        n_neighbors=n_neighbors
    )

    recommendations = []

    seed_artist_id = df_clean.iloc[seed_position]["artist_id"]

    for distance, idx in zip(distances[0], indices[0]):
        row = df_clean.iloc[idx]

        if row["artist_id"] == seed_artist_id:
            continue

        if row["artist_id"] in blacklist_artist_ids:
            continue

        if row["tag_count_clean"] < min_tag_count:
            continue

        if genre_filter is not None:
            genre_filter_lower = genre_filter.lower()

            genres = str(row["genres"]).lower()
            tags = str(row["tags"]).lower()

            if genre_filter_lower not in genres and genre_filter_lower not in tags:
                continue

        recommendations.append({
            "artist_id": int(row["artist_id"]),
            "artist_name": row["artist_name"],
            "artist_type": row["artist_type"],
            "area_name": row["area_name"],
            "genres": row["genres"],
            "tags": row["tags"],
            "tag_count_sum": int(row["tag_count_sum"]),
            "similarity_score": round(1 - distance, 4)
        })

        if len(recommendations) >= top_n:
            break

    return recommendations

In [24]:
result_1 = recommend_artists(
    artist_name="Eminem",
    top_n=5
)

result_1

[{'artist_id': 1341893,
  'artist_name': 'Conway',
  'artist_type': 'Person',
  'area_name': 'Buffalo',
  'genres': 'boom bap conscious hip hop east coast hip hop gangsta rap hardcore hip hop hip hop',
  'tags': 'boom bap conscious hip hop east coast hip hop gangsta rap hardcore hip hop hip hop',
  'tag_count_sum': 12,
  'similarity_score': np.float64(0.7615)},
 {'artist_id': 2052342,
  'artist_name': 'Cryptonyte',
  'artist_type': 'Person',
  'area_name': 'Amarillo',
  'genres': 'conscious hip hop gangsta rap hardcore hip hop hip hop underground hip hop',
  'tags': 'conscious hip hop gangsta rap hardcore hip hop hip hop underground hip hop',
  'tag_count_sum': 5,
  'similarity_score': np.float64(0.7464)},
 {'artist_id': 2504,
  'artist_name': 'Wu‐Tang Clan',
  'artist_type': 'Group',
  'area_name': 'United States',
  'genres': 'boom bap conscious hip hop east coast hip hop gangsta rap hardcore hip hop hip hop instrumental hip hop',
  'tags': 'boom bap conscious hip hop east coast hip 

Avec artist_id en input

In [25]:
df = pd.read_csv("../raw_data/artist_reco_dataset_sample.csv")

df_clean = df.copy()

# Remplacement des valeurs manquantes utiles
df_clean["artist_name"] = df_clean["artist_name"].fillna("")
df_clean["genres"] = df_clean["genres"].fillna("")

# Suppression des doublons
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset=["artist_id"])

# Retirer Various Artists si présent
df_clean = df_clean[
    df_clean["artist_name"].str.lower() != "various artists"
].copy()

# Garder uniquement les artistes avec au moins un genre
df_clean = df_clean[
    df_clean["genres"].str.strip() != ""
].copy()

artist_names = df_clean["artist_name"].tolist()

df_clean.shape

(202409, 7)

In [26]:
df_clean[["artist_id", "artist_name", "genres"]].head()

,artist_id,artist_name,genres
1,4,Massive Attack,alternative dance alternative rock ambient chi...
2,6,Apartment 26,alternative metal electronic jazz metal nu metal
4,9,Robert Miles,ambient ambient house downtempo dream trance e...
6,11,Squirrel Nut Zippers,jazz swing swing revival
7,12,Giant Sand,alternative country alternative rock country d...


In [27]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=None,
    ngram_range=(1, 2),
    min_df=2
)

X = vectorizer.fit_transform(df_clean["genres"])

X.shape

(202409, 5000)

In [28]:
knn_model = NearestNeighbors(
    n_neighbors=20,
    metric="cosine",
    algorithm="brute"
)

knn_model.fit(X)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",20
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [29]:
def recommend_artists_by_artist(
    artist_name=None,
    artist_id=None,
    top_n=5,
    genre_filter=None,
    blacklist_artist_ids=None
):
    """
    Recommande des artistes proches à partir d'un artiste de départ.

    Entrée possible :
    - artist_name
    - artist_id

    Similarité calculée uniquement à partir de la colonne genres.
    """

    if blacklist_artist_ids is None:
        blacklist_artist_ids = []

    # Recherche de l'artiste de départ
    if artist_id is not None:
        matches = df_clean[df_clean["artist_id"] == artist_id]
    elif artist_name is not None:
        matches = df_clean[
            df_clean["artist_name"].str.lower() == artist_name.lower()
        ]
    else:
        raise ValueError("Il faut fournir artist_name ou artist_id.")

    if matches.empty:
        return []

    seed_index = matches.index[0]
    seed_position = df_clean.index.get_loc(seed_index)

    query_vector = X[seed_position]

    n_neighbors = min(len(df_clean), top_n + 50)

    distances, indices = knn_model.kneighbors(
        query_vector,
        n_neighbors=n_neighbors
    )

    recommendations = []

    seed_artist_id = df_clean.iloc[seed_position]["artist_id"]

    for distance, idx in zip(distances[0], indices[0]):
        row = df_clean.iloc[idx]

        # Exclure l'artiste de départ
        if row["artist_id"] == seed_artist_id:
            continue

        # Exclure les artistes blacklistés
        if row["artist_id"] in blacklist_artist_ids:
            continue

        # Filtre optionnel sur genre
        if genre_filter is not None:
            genre = genre_filter.lower()
            genres = str(row["genres"]).lower()

            if genre not in genres:
                continue

        result = {
            "artist_id": int(row["artist_id"]),
            "artist_name": row["artist_name"],
            "genres": row["genres"],
            "similarity_score": round(1 - distance, 4)
        }

        # Ajouter artist_gid si la colonne existe
        if "artist_gid" in df_clean.columns:
            result["artist_gid"] = row["artist_gid"]

        # Ajouter d'autres infos si elles existent
        if "artist_type" in df_clean.columns:
            result["artist_type"] = row["artist_type"]

        if "area_name" in df_clean.columns:
            result["area_name"] = row["area_name"]

        recommendations.append(result)

        if len(recommendations) >= top_n:
            break

    return recommendations

In [30]:
from pathlib import Path
import os
import joblib
from google.cloud import storage

Path("../models").mkdir(exist_ok=True)
model_path = Path("../models/knn_baseline_model.pkl")


artifact = {
    "vectorizer": vectorizer,
    "model": knn_model,
    "artist_names": artist_names,
    "data": df_clean
}

joblib.dump(artifact, model_path)

bucket_name = os.getenv("MODEL_BUCKET_NAME")
blob_name = os.getenv("MODEL_BLOB_NAME", "models/knn_baseline_model.pkl")

if bucket_name:
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(model_path)
    print(f"Model uploaded to gs://{bucket_name}/{blob_name}")
else:
    print(f"Model saved locally at {model_path}")

Model saved locally at ../models/knn_baseline_model.pkl


In [31]:
recommend_artists_by_artist(
    artist_name="Eminem",
    top_n=5
)

[{'artist_id': 269151,
  'artist_name': 'Doomsday Productions',
  'genres': 'gangsta rap hardcore hip hop hip hop horrorcore',
  'similarity_score': np.float64(0.7787),
  'artist_type': 'Group',
  'area_name': nan},
 {'artist_id': 1341893,
  'artist_name': 'Conway',
  'genres': 'boom bap conscious hip hop east coast hip hop gangsta rap hardcore hip hop hip hop',
  'similarity_score': np.float64(0.7773),
  'artist_type': 'Person',
  'area_name': 'Buffalo'},
 {'artist_id': 2504,
  'artist_name': 'Wu‐Tang Clan',
  'genres': 'boom bap conscious hip hop east coast hip hop gangsta rap hardcore hip hop hip hop instrumental hip hop',
  'similarity_score': np.float64(0.7592),
  'artist_type': 'Group',
  'area_name': 'United States'},
 {'artist_id': 2052342,
  'artist_name': 'Cryptonyte',
  'genres': 'conscious hip hop gangsta rap hardcore hip hop hip hop underground hip hop',
  'similarity_score': np.float64(0.7554),
  'artist_type': 'Person',
  'area_name': 'Amarillo'},
 {'artist_id': 2091273,

In [32]:
recommend_artists_by_artist(
    artist_name="Eminem",
    genre_filter="hip hop",
    top_n=5
)

[{'artist_id': 269151,
  'artist_name': 'Doomsday Productions',
  'genres': 'gangsta rap hardcore hip hop hip hop horrorcore',
  'similarity_score': np.float64(0.7787),
  'artist_type': 'Group',
  'area_name': nan},
 {'artist_id': 1341893,
  'artist_name': 'Conway',
  'genres': 'boom bap conscious hip hop east coast hip hop gangsta rap hardcore hip hop hip hop',
  'similarity_score': np.float64(0.7773),
  'artist_type': 'Person',
  'area_name': 'Buffalo'},
 {'artist_id': 2504,
  'artist_name': 'Wu‐Tang Clan',
  'genres': 'boom bap conscious hip hop east coast hip hop gangsta rap hardcore hip hop hip hop instrumental hip hop',
  'similarity_score': np.float64(0.7592),
  'artist_type': 'Group',
  'area_name': 'United States'},
 {'artist_id': 2052342,
  'artist_name': 'Cryptonyte',
  'genres': 'conscious hip hop gangsta rap hardcore hip hop hip hop underground hip hop',
  'similarity_score': np.float64(0.7554),
  'artist_type': 'Person',
  'area_name': 'Amarillo'},
 {'artist_id': 2091273,